# Embeddings and Contrastive Learning: Making Similar Mean Close

The runnable code of [Embeddings and Contrastive Learning: Making Similar Mean Close](https://residual-stream.lvtan71.workers.dev/posts/embeddings-and-contrastive-learning/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/embeddings-and-contrastive-learning.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 2 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "embeddings-and-contrastive-learning"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/embeddings-and-contrastive-learning/sections.py', 'shared/embeddings-and-contrastive-learning/split.py', 'data/blog_train.txt', 'data/blog_val.txt', 'data/blog-47b01b1.tar.gz']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Word2vec: meaning from neighbors

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/embeddings-and-contrastive-learning/#word2vec-meaning-from-neighbors)

In [ ]:
%%writefile step0.py
import re
import sys
from collections import Counter
import torch
from split import train_text

# Skip-gram with negative sampling (Mikolov et al., 2013) on this blog's
# words: learn a vector per word such that words appearing near each other
# have a large dot product, and random pairs a small one. Trained only on
# text outside the held-out sections. Two simplifications from the original:
# frequent words aren't subsampled, and every distance in the window counts
# equally.
seed = int(sys.argv[1]) if len(sys.argv) > 1 else 0
text = train_text.lower()
words = re.findall(r"[a-z][a-z0-9\-]+", text)
counts = Counter(words)
vocab = [w for w, c in counts.items() if c >= 5]
index = {w: i for i, w in enumerate(vocab)}
ids = torch.tensor([index[w] for w in words if w in index])
print(f"{len(words):,} words, {len(vocab):,} kept (5+ occurrences)")

torch.manual_seed(seed)
D, window, k = 64, 4, 5
center_emb = torch.nn.Embedding(len(vocab), D)
context_emb = torch.nn.Embedding(len(vocab), D)
torch.nn.init.normal_(center_emb.weight, 0, 0.1); torch.nn.init.zeros_(context_emb.weight)
freq = torch.tensor([counts[w] for w in vocab], dtype=torch.float) ** 0.75   # word2vec's sampling rule
opt = torch.optim.Adam(list(center_emb.parameters()) + list(context_emb.parameters()), lr=3e-3)
for step in range(4000):
    pos = torch.randint(window, len(ids) - window, (1024,))
    offset = torch.randint(1, window + 1, (1024,)) * (torch.randint(0, 2, (1024,)) * 2 - 1)
    c, o = ids[pos], ids[pos + offset]                                # a real (center, context) pair
    neg = torch.multinomial(freq, 1024 * k, replacement=True).view(1024, k)  # k random "contexts"
    vc = center_emb(c)
    pos_score = (vc * context_emb(o)).sum(-1)
    neg_score = torch.einsum("bd,bkd->bk", vc, context_emb(neg))
    # Logistic loss ("Learning Is Optimization"): real pairs toward 1, random pairs toward 0.
    loss = -(torch.nn.functional.logsigmoid(pos_score).mean()
             + torch.nn.functional.logsigmoid(-neg_score).sum(-1).mean())
    opt.zero_grad(); loss.backward(); opt.step()
print(f"final loss {loss.item():.3f} (at the start: (1 + k) log 2 = {(1 + k) * 0.6931:.2f})")

raw = center_emb.weight.detach()
E = torch.nn.functional.normalize(raw, dim=1)                          # unit length: dot = cosine
torch.save({"vocab": vocab, "E": E, "raw": raw, "counts": [counts[w] for w in vocab]},
           f"word2vec{'' if seed == 0 else seed}.pt")
for w in ("attention", "cache", "gradient", "token", "vietnamese", "gpu"):
    sims = E @ E[index[w]]
    best = [vocab[j] for j in sims.argsort(descending=True)[1:7]]
    print(f"{w:11s} -> {', '.join(best)}")
for a, b in (("cache", "caching"), ("token", "tokens"), ("cache", "cached")):
    rank = (E @ E[index[a]]).argsort(descending=True).tolist().index(index[b])
    print(f"cos({a}, {b}) = {(E[index[a]] @ E[index[b]]).item():.2f}, rank {rank} of {len(vocab):,}")

In [ ]:
# The post trained three seeds; the next cells compare them.
%run step0.py 0
%run step0.py 1
%run step0.py 2

The post printed:

```text
60,355 words, 1,685 kept (5+ occurrences)
final loss 2.071 (at the start: (1 + k) log 2 = 4.16)
attention   -> sparse, multi-head, hybrid, transformer, flashattention, gqa
cache       -> kv, kv-cache-explained, linearly, per-sequence, quantization, iteration-level
gradient    -> descent, alive, mse, convex, curvature, failures
token       -> representation, accepted, dimensional, speculative, decoder, single
vietnamese  -> english, questions, route, reader, foundations, list
gpu         -> h100, datasheet, laptop, tflops, cores, bandwidth
cos(cache, caching) = 0.40, rank 71 of 1,685
cos(token, tokens) = 0.49, rank 14 of 1,685
cos(cache, cached) = 0.39, rank 89 of 1,685
```

In [ ]:
from collections import Counter
import torch

# Which neighbours are stable? Top 6 by cosine in each of three word2vec seeds.
seeds = [torch.load(f"word2vec{'' if s == 0 else s}.pt") for s in (0, 1, 2)]
for w in ("attention", "cache", "gradient", "token", "vietnamese", "gpu"):
    c = Counter()
    for m in seeds:
        vocab, E = m["vocab"], m["E"]
        i = vocab.index(w)
        c.update(vocab[j] for j in (E @ E[i]).argsort(descending=True)[1:7].tolist())
    print(f"{w:11s} in all 3 seeds: {', '.join(x for x, n in c.items() if n == 3) or '-'};  "
          f"in 2: {', '.join(x for x, n in c.items() if n == 2) or '-'}")

The post printed:

```text
attention   in all 3 seeds: sparse, multi-head, hybrid;  in 2: transformer
cache       in all 3 seeds: kv, kv-cache-explained, per-sequence;  in 2: linearly
gradient    in all 3 seeds: descent, alive, mse;  in 2: convex, failures
token       in all 3 seeds: representation;  in 2: accepted, decoder, re-read
vietnamese  in all 3 seeds: english, questions;  in 2: -
gpu         in all 3 seeds: h100, datasheet;  in 2: laptop, tflops, bandwidth
```

In [ ]:
import torch

# The famous test: does b - a + c land nearest to d? ("king - man + woman ~ queen")
m = torch.load("word2vec.pt")
vocab, E = m["vocab"], m["E"]
index = {w: i for i, w in enumerate(vocab)}
tests = [("encoder", "decoder", "input", "output"), ("keys", "values", "key", "value"),
         ("prefill", "decode", "prompt", "generation"),
         ("layer", "layers", "token", "tokens"), ("head", "heads", "weight", "weights"),
         ("key", "keys", "value", "values")]
for a, b, c, d in tests:
    if not all(w in index for w in (a, b, c, d)):
        print(f"{a}:{b} :: {c}:{d}  (a word is missing from the vocabulary)")
        continue
    q = E[index[b]] - E[index[a]] + E[index[c]]
    sims = torch.nn.functional.normalize(q, dim=0) @ E.T
    for w in (a, b, c):
        sims[index[w]] = -1                          # the usual rule: exclude the question words
    ranked = sims.argsort(descending=True)
    rank = (ranked == index[d]).nonzero().item() + 1
    print(f"{a} : {b} :: {c} : ?   top answer '{vocab[ranked[0]]}', the expected '{d}' is ranked {rank} "
          f"of {len(vocab) - 3:,}")

The post printed:

```text
encoder : decoder :: input : ?   top answer 'trigger', the expected 'output' is ranked 121 of 1,682
keys : values :: key : ?   top answer 'self', the expected 'value' is ranked 36 of 1,682
prefill : decode :: prompt : ?   top answer 'character', the expected 'generation' is ranked 168 of 1,682
layer : layers :: token : ?   top answer 'speculative', the expected 'tokens' is ranked 91 of 1,682
head : heads :: weight : ?   top answer 'neurips', the expected 'weights' is ranked 15 of 1,682
key : keys :: value : ?   top answer 're-read', the expected 'values' is ranked 9 of 1,682
```

In [ ]:
import torch

# Dot product vs cosine on the raw word2vec vectors. The dot product is the
# cosine times both lengths, so a long vector rises in any list it's already
# near.
m = torch.load("word2vec.pt")
vocab, raw, counts = m["vocab"], m["raw"], torch.tensor(m["counts"], dtype=torch.float)
index = {w: i for i, w in enumerate(vocab)}
norms = raw.norm(dim=1)
r = torch.corrcoef(torch.stack([norms, counts.log()]))[0, 1]
print(f"vector length vs log word frequency: correlation {r:.2f}")
q = raw[index["cache"]]
for name, scores in [("dot product", raw @ q), ("cosine", (raw @ q) / (norms * q.norm()))]:
    scores[index["cache"]] = -1e9
    top = scores.argsort(descending=True)[:6]
    print(f"nearest to 'cache' by {name:11s}: " + ", ".join(f"{vocab[j]} ({norms[j]:.1f})" for j in top))
print(f"(vector lengths in brackets; the median word's is {norms.median():.1f})")

The post printed:

```text
vector length vs log word frequency: correlation -0.20
nearest to 'cache' by dot product: kv (4.9), xhat (7.1), kv-cache-explained (5.4), nh (5.7), color-background (7.9), fill (7.4)
nearest to 'cache' by cosine     : kv (4.9), kv-cache-explained (5.4), linearly (4.6), per-sequence (4.9), quantization (3.8), iteration-level (4.9)
(vector lengths in brackets; the median word's is 4.3)
```

## InfoNCE: a cross-entropy over one right answer

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/embeddings-and-contrastive-learning/#infonce-a-cross-entropy-over-one-right-answer)

In [ ]:
import math
import torch

def info_nce(a, b, tau):
    """Row i of a should match row i of b and no other row: cross-entropy over
    the similarity matrix, with the diagonal as the correct 'class' ("Cross-Entropy and the Softmax Gradient")."""
    a, b = torch.nn.functional.normalize(a, dim=1), torch.nn.functional.normalize(b, dim=1)
    logits = a @ b.T / tau
    return torch.nn.functional.cross_entropy(logits, torch.arange(len(a))), a @ b.T

# Three pairs of 2-D embeddings: each b is a slightly rotated copy of its a.
a = torch.tensor([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.2]])
b = torch.tensor([[0.9, 0.3], [0.2, 1.0], [-1.0, -0.1]])
loss, sim = info_nce(a, b, 1.0)
print("cosine similarities (rows: a, columns: b):")
for row in sim:
    print("   " + "  ".join(f"{v:+.3f}" for v in row))
for tau in (1.0, 0.5, 0.1, 0.05):
    print(f"temperature {tau:<4}: loss {info_nce(a, b, tau)[0]:.4f}")

# Random, unrelated embeddings: the loss is about log(batch size), a uniform
# guess among B ("Cross-Entropy and the Softmax Gradient"), and the best possible loss is 0. The bound:
# mutual information >= log B - loss. (At a low temperature, random
# similarities spread out and the loss goes above log B.)
torch.manual_seed(0)
for B in (4, 64, 1024):
    x, y = torch.randn(B, 64), torch.randn(B, 64)
    print(f"batch {B:5d}: random embeddings give loss {info_nce(x, y, 1.0)[0]:.2f} at temperature 1, "
          f"{info_nce(x, y, 0.1)[0]:.2f} at 0.1; log B = {math.log(B):.2f}")

The post printed:

```text
cosine similarities (rows: a, columns: b):
   +0.949  +0.196  -0.995
   +0.316  +0.981  -0.100
   -0.868  +0.000  +0.956
temperature 1.0 : loss 0.5106
temperature 0.5 : loss 0.2332
temperature 0.1 : loss 0.0006
temperature 0.05: loss 0.0000
batch     4: random embeddings give loss 1.43 at temperature 1, 2.11 at 0.1; log B = 1.39
batch    64: random embeddings give loss 4.17 at temperature 1, 4.93 at 0.1; log B = 4.16
batch  1024: random embeddings give loss 6.94 at temperature 1, 7.68 at 0.1; log B = 6.93
```

## A contrastive head on a tiny corpus

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/embeddings-and-contrastive-learning/#a-contrastive-head-on-a-tiny-corpus)

In [ ]:
import re
import torch

# Split the blog into sections of raw words once: 30% of sections are held
# out for testing and never trained on, not even by word2vec. Everything
# else, sections and the text between them, is training text.
VIEW = 40                                                      # words per view
text = open("blog_train.txt").read() + open("blog_val.txt").read()
chunks = re.split(r"\n## ", text)
long_enough = [i for i, c in enumerate(chunks) if len(c.split()) >= 3 * VIEW]
order = torch.randperm(len(long_enough), generator=torch.Generator().manual_seed(0))
held_out = {long_enough[i] for i in order[:len(long_enough) * 3 // 10]}
test_secs = [chunks[i].split() for i in sorted(held_out)]
train_secs = [chunks[i].split() for i in long_enough if i not in held_out]
train_text = "\n## ".join(c for i, c in enumerate(chunks) if i not in held_out)

In [ ]:
import re
import torch
from split import train_secs, test_secs, VIEW

# The held-out split from split.py, plus word2vec vectors (trained without
# the held-out sections) and helpers shared by contrastive.py and pretrained.py.
def load(seed=0):
    m = torch.load(f"word2vec{'' if seed == 0 else seed}.pt")
    return m["vocab"], m["E"]

vocab, W2V = load(0)
index = {w: i for i, w in enumerate(vocab)}

def to_ids(toks, index=index):
    """The words word2vec knows, in order (vocab[0] if there are none)."""
    ids = [index[w] for t in toks for w in re.findall(r"[a-z][a-z0-9\-]+", t.lower()) if w in index]
    return torch.tensor(ids or [0])

def crop(sec, g):
    s = torch.randint(0, len(sec) - VIEW + 1, (1,), generator=g).item()
    return sec[s:s + VIEW]

In [ ]:
import torch

# Learn to embed a passage so that two pieces of the same blog section land
# close together: SimCLR's idea (two "views" of one thing are a positive pair,
# everything else in the batch is a negative), with random crops as the views.
from sections import W2V, load, train_secs, test_secs, crop, to_ids, VIEW

class Encoder(torch.nn.Module):
    """Average the (frozen) word2vec vectors, then a learned linear map."""
    def __init__(self, init):
        super().__init__()
        self.emb = torch.nn.Embedding.from_pretrained(init.clone(), freeze=True)
        self.proj = torch.nn.Linear(init.shape[1], init.shape[1])
        torch.nn.init.eye_(self.proj.weight); torch.nn.init.zeros_(self.proj.bias)
    def forward(self, windows):                                # a list of word windows
        pooled = torch.stack([self.emb(to_ids(w)).mean(0) for w in windows])
        return torch.nn.functional.normalize(self.proj(pooled), dim=-1)

@torch.no_grad()
def recall_at_1(encode, secs=test_secs):
    """The first 40 words of each section should find that section's last 40
    words among all the sections' last 40 words."""
    a = encode([s[:VIEW] for s in secs])
    b = encode([s[-VIEW:] for s in secs])
    return ((a @ b.T).argmax(1) == torch.arange(len(a))).float().mean().item()

mean_of = lambda E: lambda ws: torch.nn.functional.normalize(
    torch.stack([E[to_ids(w)].mean(0) for w in ws]), dim=-1)
torch.manual_seed(0)
random_emb = torch.nn.functional.normalize(torch.randn_like(W2V), dim=1)
print(f"{len(train_secs)} training sections, {len(test_secs)} held-out sections")
print(f"before training, the encoder is exactly the word2vec average: recall@1 {recall_at_1(Encoder(W2V)):.3f}")
print(f"random word vectors, averaged:   recall@1 {recall_at_1(mean_of(random_emb)):.3f}")
w2v = []
for seed in (0, 1, 2):
    vocab_s, E_s = load(seed)
    idx = {w: i for i, w in enumerate(vocab_s)}
    w2v.append(recall_at_1(lambda ws: torch.nn.functional.normalize(
        torch.stack([E_s[to_ids(w, idx)].mean(0) for w in ws]), dim=-1)))
print("word2vec vectors, averaged:      recall@1 " + ", ".join(f"{v:.3f}" for v in w2v)
      + " (3 word2vec seeds)")

for B in (4, 16, 64):
    scores, fit = [], []
    for seed in range(3):
        g = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
        enc = Encoder(W2V)
        opt = torch.optim.Adam(enc.parameters(), lr=1e-3)
        for step in range(1500):
            pick = torch.randint(0, len(train_secs), (B,), generator=g)
            x1 = [crop(train_secs[i], g) for i in pick]
            x2 = [crop(train_secs[i], g) for i in pick]
            logits = enc(x1) @ enc(x2).T / 0.1                  # InfoNCE, temperature 0.1
            same = pick[:, None] == pick[None, :]               # repeats in a batch aren't negatives
            logits = logits.masked_fill(same & ~torch.eye(B, dtype=torch.bool), float("-inf"))
            loss = torch.nn.functional.cross_entropy(logits, torch.arange(B))
            opt.zero_grad(); loss.backward(); opt.step()
        scores.append(recall_at_1(enc))
        fit.append(recall_at_1(enc, train_secs[:len(test_secs)]))   # same number of training sections
    print(f"contrastive, batch {B:3d} ({B - 1:2d} negatives): recall@1 "
          + ", ".join(f"{s:.3f}" for s in scores) + " (3 seeds); on training sections "
          + ", ".join(f"{s:.3f}" for s in fit))

The post printed:

```text
156 training sections, 66 held-out sections
before training, the encoder is exactly the word2vec average: recall@1 0.152
random word vectors, averaged:   recall@1 0.061
word2vec vectors, averaged:      recall@1 0.152, 0.136, 0.136 (3 word2vec seeds)
contrastive, batch   4 ( 3 negatives): recall@1 0.197, 0.152, 0.212 (3 seeds); on training sections 0.121, 0.121, 0.121
contrastive, batch  16 (15 negatives): recall@1 0.167, 0.182, 0.212 (3 seeds); on training sections 0.136, 0.152, 0.136
contrastive, batch  64 (63 negatives): recall@1 0.212, 0.227, 0.212 (3 seeds); on training sections 0.182, 0.182, 0.167
```

## Pretrained sentence encoders

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/embeddings-and-contrastive-learning/#pretrained-sentence-encoders)

In [ ]:
import glob
import os
import re
import time
import torch
from sentence_transformers import SentenceTransformer
from sections import test_secs, W2V, to_ids, VIEW

# The same held-out task as contrastive.py, with two pretrained sentence
# encoders: all-MiniLM-L6-v2, trained contrastively on about a billion
# English sentence pairs, and a multilingual model distilled from an English
# one on translated pairs (Reimers and Gurevych, 2020).
models = {"all-MiniLM-L6-v2 (English)": "sentence-transformers/all-MiniLM-L6-v2",
          "multilingual MiniLM-L12": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"}
first = [" ".join(s[:VIEW]) for s in test_secs]           # raw text: the same words the others see
last = [" ".join(s[-VIEW:]) for s in test_secs]

# Sections of the posts that have a Vietnamese twin, paired by position.
POSTS = "content"
def split(path):
    text = re.sub(r"\A---.*?---\n", "", open(path).read(), flags=re.S)
    skip = ("Table of contents", "Mục lục", "References", "Tài liệu tham khảo")   # headings only, or citation lists
    return [s.strip()[:1500] for s in re.split(r"\n## ", text)[1:] if not s.startswith(skip)]
pairs = []
for vi_path in sorted(glob.glob(f"{POSTS}/posts-vi/*.mdx")):
    en, vi = split(f"{POSTS}/posts/{os.path.basename(vi_path)}"), split(vi_path)
    if len(en) == len(vi):                                                # only posts whose headings line up
        pairs += list(zip(en, vi))
print(f"{len(test_secs)} held-out English sections; {len(pairs)} English/Vietnamese section pairs")

def recall(model, queries, docs):
    q = model.encode(queries, normalize_embeddings=True, convert_to_tensor=True)
    d = model.encode(docs, normalize_embeddings=True, convert_to_tensor=True)
    return ((q @ d.T).argmax(1) == torch.arange(len(q), device=q.device)).float().mean().item()

def spread(E):
    """Average cosine between *different* held-out English windows."""
    S = E @ E.T
    return S[~torch.eye(len(E), dtype=torch.bool)].mean().item()

w2v = torch.nn.functional.normalize(torch.stack([W2V[to_ids(s[:VIEW])].mean(0) for s in test_secs]), dim=-1)
print(f"averaged word2vec: average cosine between different held-out windows {spread(w2v):.2f}")
for name, path in models.items():
    model = SentenceTransformer(path, device="cpu")
    params = sum(p.numel() for p in model.parameters())
    same = recall(model, first, last)
    t0 = time.perf_counter()
    cross = recall(model, [e for e, _ in pairs], [v for _, v in pairs])
    secs = time.perf_counter() - t0
    E = model.encode(first, normalize_embeddings=True, convert_to_tensor=True)
    print(f"{name}: {params / 1e6:.0f}M parameters, reads up to {model.max_seq_length} tokens")
    print(f"    same-section recall@1 {same:.3f}   English -> Vietnamese recall@1 {cross:.3f} "
          f"({2 * len(pairs)} sections in {secs:.1f} s)   average cosine between different windows {spread(E):.2f}")

The post printed:

```text
66 held-out English sections; 118 English/Vietnamese section pairs
averaged word2vec: average cosine between different held-out windows 0.81
all-MiniLM-L6-v2 (English): 23M parameters, reads up to 256 tokens
    same-section recall@1 0.318   English -> Vietnamese recall@1 0.576 (236 sections in 2.0 s)   average cosine between different windows 0.16
multilingual MiniLM-L12: 118M parameters, reads up to 128 tokens
    same-section recall@1 0.212   English -> Vietnamese recall@1 1.000 (236 sections in 2.0 s)   average cosine between different windows 0.29
```